In [0]:
from pyspark.sql import functions as F
from collections import Counter

project_path = "/Volumes/workspace/default/freddie_mac_raw/DB Project/"

# Find the text files, including files inside year folders.
def find_text_files(folder):
    files = []

    for item in dbutils.fs.ls(folder):
        if item.isDir():
            files.extend(find_text_files(item.path))
        elif item.name.endswith(".txt"):
            files.append(item)

    return files

project_files = find_text_files(project_path)

expected_files = {
    f"sample_{kind}_{year}.txt"
    for year in range(2016, 2026)
    for kind in ["orig", "perf"]
}

file_counts = Counter(file.name for file in project_files)

missing_files = sorted(expected_files - set(file_counts))
unexpected_files = sorted(set(file_counts) - expected_files)
duplicate_files = sorted(
    name for name, count in file_counts.items() if count > 1
)
empty_files = sorted(file.name for file in project_files if file.size == 0)

assert not missing_files, f"Missing files: {missing_files}"
assert not unexpected_files, f"Unexpected files: {unexpected_files}"
assert not duplicate_files, f"Duplicate files: {duplicate_files}"
assert not empty_files, f"Empty files: {empty_files}"

print(f"Validated {len(project_files)} source files.")


Validated 20 source files.


In [0]:
# Define the columns in file order
orig_columns = [
    "fico", "first_payment_date", "first_time_homebuyer",
    "maturity_date", "msa", "mi_pct", "units", "occupancy",
    "cltv", "dti", "original_upb", "ltv", "original_rate",
    "channel", "prepayment_penalty", "amortization_type",
    "state", "property_type", "postal_code", "loan_id",
    "purpose", "original_term", "borrowers", "seller",
    "super_conforming", "pre_harp_id", "eligibility_program",
    "harp", "valuation_method", "interest_only", "vantage_score"
]

perf_columns = [
    "loan_id", "period", "current_upb", "delinquency",
    "loan_age", "remaining_term", "defect_date", "modification",
    "zero_balance_code", "zero_balance_date", "current_rate",
    "deferred_upb", "ddlpi", "mi_recoveries", "net_sale_proceeds",
    "non_mi_recoveries", "expenses", "legal_costs",
    "maintenance_costs", "taxes_insurance",
    "miscellaneous_expenses", "actual_loss",
    "cumulative_modification_cost", "step_indicator",
    "deferral", "estimated_ltv", "removal_upb",
    "delinquent_interest", "disaster", "assistance",
    "current_modification_cost", "interest_bearing_upb",
    "mi_cancellation", "servicer", "bankruptcy_cost"
]

In [0]:
# Define the Bronze loading function
def build_bronze(kind, columns):
    paths = sorted(
        file.path
        for file in project_files
        if file.name.startswith(f"sample_{kind}_")
    )

    lines = (
        spark.read.text(paths)
        .select(
            F.col("value").alias("raw_line"),
            F.col("_metadata.file_name").alias("source_file"),
            F.col("_metadata.file_path").alias("source_path")
        )
        .withColumn("_fields", F.split(F.col("raw_line"), r"\|", -1))
    )

    # Check the structure before saving anything.
    invalid_rows = lines.filter(
        (F.size("_fields") != len(columns))
        | (F.trim(F.col("raw_line")) == "")
    )

    assert invalid_rows.limit(1).count() == 0, (
        f"Invalid row structure found in the {kind} files. "
        f"Each row should have {len(columns)} fields."
    )

    return lines.select(
        *[
            F.col("_fields").getItem(i).alias(name)
            for i, name in enumerate(columns)
        ],
        "raw_line",
        "source_file",
        "source_path",
        F.regexp_extract(
            F.col("source_file"), r"_(\d{4})\.txt$", 1
        ).cast("int").alias("source_vintage"),
        F.current_timestamp().alias("ingested_at")
    )


bronze_orig = build_bronze("orig", orig_columns)
bronze_perf = build_bronze("perf", perf_columns)

print("Both datasets passed the row-structure checks.")


Both datasets passed the row-structure checks.


In [0]:
# Build and refresh both tables
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.mortgage_risk")

# Rebuild both tables from the complete set of source files.
(
    bronze_orig.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.mortgage_risk.bronze_origination")
)

(
    bronze_perf.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.mortgage_risk.bronze_performance")
)

print("Both Bronze tables saved successfully.")


Both Bronze tables saved successfully.


In [0]:
# Check the saved tables
summary = spark.sql("""
    SELECT
        'origination' AS table_name,
        COUNT(*) AS row_count,
        COUNT(DISTINCT source_file) AS source_files
    FROM workspace.mortgage_risk.bronze_origination

    UNION ALL

    SELECT
        'performance' AS table_name,
        COUNT(*) AS row_count,
        COUNT(DISTINCT source_file) AS source_files
    FROM workspace.mortgage_risk.bronze_performance
""")

display(summary)


table_name,row_count,source_files
origination,500000,10
performance,20079736,10
